# Training YOLO11n ourselves on all 80 COCO classes: from scratch (B) and fine-tuned (C)

Kontur suggests objects with **YOLO11n pretrained on COCO** by Ultralytics (model **A**,
`weights/yolo11n.pt`). This notebook trains the **same network ourselves**, on the **same
80 classes**, so all three models do exactly the same job:

| | Model | Weights at the start |
|---|---|---|
| **B** | **YOLO11n from scratch** | **random** (built from the architecture file `yolo11n.yaml`) |
| **C** | YOLO11n fine-tuned | a copy of Kontur's `weights/yolo11n.pt` (that file is only read) |

B and C get the same data, epochs, batch size, image size, seed and settings; the **only**
difference is the starting point. The comparison with A is in
**`eval/04_scratch_vs_pretrained.ipynb`**; this notebook builds the data and creates the models.

**Data:** COCO 2017 [Lin et al. 2014], downloaded by this notebook.

| Split | From | Used for |
|---|---|---|
| `val` | 1,000 random train2017 images (chosen first) | picking the best epoch |
| `train` | every other train2017 image with an object (about 116k) | training B and C |
| `test` | all 5,000 val2017 images: the official COCO benchmark | **only** the final comparison (`eval/04`) |

**How to run, in order:**
1. *Settings* to *Step 4*: data (downloads only what is missing; about 18 GB in total).
2. *Quick check* (optional, ~5 min): one tiny epoch of B and C, to catch problems before hours of GPU time.
3. *Launch*: starts B and C **in the background** (both at once on the one GPU, by default).
   They keep running if you close the browser tab.
4. *Status*: re-run any time to see the epoch, and the estimated time left.
5. *Finish*: when both are done, copies the weights and curves into `trained/` and `results/`.

If the data lab stops the runs, set `RESUME = True` and run *Launch* again: each continues
from its last saved epoch.

## Settings

In [1]:
from pathlib import Path
import os

HERE = Path.cwd()                      # the training/ folder (Jupyter runs a notebook from its own folder)
if not (HERE / "train_yolo11n_coco.ipynb").exists() and (HERE / "training").is_dir():
    HERE = HERE / "training"           # started from the repository root instead
ROOT = HERE.parent                     # the repository root

# Where COCO goes. Default: training/data. Any path works, or set the environment
# variable KONTUR_DATA (eval/04 reads the same variable, so both find the data).
DATA_DIR = Path(os.environ.get("KONTUR_DATA", HERE / "data")).resolve()

CLASSES = None              # None = all 80 COCO classes, read from the annotations
MAX_TRAIN_IMAGES = None     # None = every train2017 image with an object (~116k); a number = a random subset
VAL_IMAGES = 1_000          # held-out train2017 images, for picking the best epoch
EPOCHS = 50                 # the same for B and C. 50 fits the deadline on one A6000 (see Launch)
PATIENCE = EPOCHS           # no early stopping: B and C both get every epoch
IMGSZ = 640
BATCH = 64                  # FIXED and the same for B and C (AutoBatch could pick different sizes)
WORKERS = 0                 # 0: the data lab container's /dev/shm is only 64 MB, too small for
                            # data-loading worker processes (each batch is ~78 MB of shared memory)
TRAIN_FRACTION = 30_000     # train on exactly 30,000 of the ~116k train images, the SAME ones for B
                            # and C (Ultralytics takes the first N files in name order; val and test
                            # stay complete). 1.0 = all images. Without workers, all would not fit the deadline.
DEVICE = 0                  # the A6000
SEED = 2026                 # same seed as Kontur's evaluation
DOWNLOAD_THREADS = 32

PARALLEL = True             # True: B and C train at the same time on the one GPU (48 GB fits both)
RUN_B = True                # train from scratch
RUN_C = True                # fine-tune from the pretrained weights
RESUME = False              # True: continue interrupted runs from their last.pt
QUICK_CHECK_FRACTION = 0.01 # the quick check trains on 1% of train for one epoch

RUNS = HERE / "runs"                    # Ultralytics logs (not committed)
RESULTS = HERE / "results"              # evidence (committed)
TRAINED = HERE / "trained"              # our trained weights (committed)
PRETRAINED = ROOT / "weights" / "yolo11n.pt"   # the exact file Kontur uses; downloaded there if missing
IMG, LBL = DATA_DIR / "images", DATA_DIR / "labels"
DATA_YAML = DATA_DIR / "coco80.yaml"           # written in Step 4
for d in (DATA_DIR, RUNS, RESULTS, TRAINED, PRETRAINED.parent):
    d.mkdir(parents=True, exist_ok=True)
print("notebook folder:", HERE)
print("data folder:    ", DATA_DIR)

notebook folder: /home/jovyan/vault/Kontur/training
data folder:     /home/jovyan/vault/Kontur/training/data


## Environment (recorded as evidence)

In [2]:
import json, platform, sys, time
import torch, ultralytics

env = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else None,
    "cpu_count": os.cpu_count(),
    "started": time.strftime("%Y-%m-%d %H:%M:%S"),
    "settings": dict(CLASSES="all 80" if CLASSES is None else CLASSES, MAX_TRAIN_IMAGES=MAX_TRAIN_IMAGES,
                     VAL_IMAGES=VAL_IMAGES, EPOCHS=EPOCHS, PATIENCE=PATIENCE, IMGSZ=IMGSZ, BATCH=BATCH,
                     WORKERS=WORKERS, TRAIN_FRACTION=TRAIN_FRACTION, SEED=SEED, DEVICE=DEVICE, PARALLEL=PARALLEL),
}
old = RESULTS / "environment.json"
if old.exists():                       # after a kernel restart: keep the original start time and records
    prev = json.loads(old.read_text())
    env = {**prev, **{k: v for k, v in env.items() if k != "started"}, "started": prev.get("started", env["started"])}
(RESULTS / "environment.json").write_text(json.dumps(env, indent=1))
print(json.dumps(env, indent=1))
if not env["cuda_available"]:
    print("\nWARNING: no GPU found. Check the data lab session has a GPU.")

{
 "python": "3.11.13",
 "platform": "Linux-6.8.0-146-generic-x86_64-with-glibc2.35",
 "torch": "2.14.1+cu130",
 "ultralytics": "8.4.187",
 "cuda_available": true,
 "gpu": "NVIDIA RTX A6000",
 "gpu_memory_gb": 50.9,
 "started": "2026-11-27 14:48:37",
 "settings": {
  "CLASSES": "all 80",
  "MAX_TRAIN_IMAGES": null,
  "VAL_IMAGES": 1000,
  "EPOCHS": 50,
  "PATIENCE": 50,
  "IMGSZ": 640,
  "BATCH": 64,
  "WORKERS": 8,
  "SEED": 2026,
  "DEVICE": 0,
  "PARALLEL": true
 },
 "cpu_count": 20
}


## Step 1: download the COCO annotations (241 MB, once)

From the official COCO site. Only the two instance files are unpacked.

In [3]:
import shutil, time, urllib.request
from pathlib import Path

UA = {"User-Agent": "Mozilla/5.0 (coco-download)"}

def fetch(url, dest, tries=4, timeout=60):
    """Download url to dest unless it is already there. Retries, then explains."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".part")
    err = None
    for k in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=timeout) as r, \
                 open(tmp, "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20)
            tmp.replace(dest)
            return dest
        except Exception as e:
            err = e
            time.sleep(2 * (k + 1))
    raise RuntimeError(f"could not download {url}: {err}")

import zipfile

ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ANN = DATA_DIR / "annotations"
NEED = ["instances_train2017.json", "instances_val2017.json"]
if not all((ANN / n).exists() for n in NEED):
    z = DATA_DIR / "annotations_trainval2017.zip"
    print("downloading", ANN_URL, "(241 MB) ...")
    try:
        fetch(ANN_URL, z, timeout=600)
    except RuntimeError as e:
        raise RuntimeError(f"{e}\nIf the data lab blocks this, download the zip in a browser and put it at {z}") from None
    with zipfile.ZipFile(z) as zf:
        for n in NEED:
            zf.extract(f"annotations/{n}", DATA_DIR)
print("annotations ready:", sorted(p.name for p in ANN.glob("*.json")))

annotations ready: ['instances_train2017.json', 'instances_val2017.json']


## Step 2: choose the images

- **Val first:** 1,000 random train2017 images with at least one object. Choosing it first means
  every class can appear in it (an earlier 5-class version drew val last, from what was left,
  and got no bicycles).
- **Train:** every other train2017 image with at least one (non-crowd) object.
- **Test:** all 5,000 val2017 images, as in the official COCO benchmark.

In [4]:
import random
from pycocotools.coco import COCO

coco_tr = COCO(str(ANN / "instances_train2017.json"))
coco_te = COCO(str(ANN / "instances_val2017.json"))
cats = sorted(coco_tr.cats.values(), key=lambda c: c["id"])     # COCO's own order = YOLO's 0..79
if CLASSES is None:
    CLASSES = [c["name"] for c in cats]
cat_ids = {c["name"]: c["id"] for c in cats if c["name"] in CLASSES}
CLASSES = list(cat_ids)                                            # keep COCO's order
cat_name = {v: k for k, v in cat_ids.items()}
assert len(CLASSES) == len(cat_ids) > 0

def with_objects(coco):
    """Images with at least one non-crowd object of the chosen classes."""
    return sorted({a["image_id"] for a in coco.anns.values()
                   if not a["iscrowd"] and a["category_id"] in cat_name})

rng = random.Random(SEED)
pool = with_objects(coco_tr)
rng.shuffle(pool)
val_ids = sorted(pool[:VAL_IMAGES])
train_ids = pool[VAL_IMAGES:]
if MAX_TRAIN_IMAGES:
    train_ids = train_ids[:MAX_TRAIN_IMAGES]
train_ids = sorted(train_ids)
test_ids = sorted(coco_te.getImgIds())               # all 5,000
split = {"train": train_ids, "val": val_ids, "test": test_ids}
SRC = {"train": coco_tr, "val": coco_tr, "test": coco_te}
assert not (set(train_ids) & set(val_ids)), "train and val overlap"

def count(coco, ids):
    ids, c = set(ids), {n: 0 for n in CLASSES}
    for a in coco.anns.values():
        if a["image_id"] in ids and not a["iscrowd"] and a["category_id"] in cat_name:
            c[cat_name[a["category_id"]]] += 1
    return c

counts = {k: count(SRC[k], v) for k, v in split.items()}
(RESULTS / "dataset.json").write_text(json.dumps(
    {"classes": CLASSES, "coco_category_ids": cat_ids, "seed": SEED,
     "images": {k: len(v) for k, v in split.items()}, "objects_per_class": counts,
     "image_ids": split}))
env["settings"]["CLASSES"] = f"{len(CLASSES)} COCO classes"
(RESULTS / "environment.json").write_text(json.dumps(env, indent=1))
for k, v in split.items():
    c = counts[k]
    print(f"{k:5s} {len(v):6d} images  {sum(c.values()):7d} objects  "
          f"classes present: {sum(1 for x in c.values() if x)}/{len(CLASSES)}  "
          f"rarest: {min(c, key=c.get)} ({min(c.values())})")

loading annotations into memory...
Done (t=10.75s)
creating index...
index created!
loading annotations into memory...
Done (t=0.29s)
creating index...
index created!
train 116266 images   842836 objects  classes present: 80/80  rarest: hair drier (196)
val     1000 images     7113 objects  classes present: 80/80  rarest: toaster (2)
test    5000 images    36335 objects  classes present: 80/80  rarest: toaster (9)


## Step 3: put every image in its split's folder, and download what is missing

YOLO trains on **every image in the train folder**, so each folder must hold exactly its split.
Images already downloaded (also by the earlier 5-class version) are moved to the folder of
their split; files that belong to no split are deleted. Only missing images are downloaded
(about 18 GB in total, parallel).

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

want = {SRC[k].imgs[i]["file_name"]: k for k, ids in split.items() for i in ids}   # file -> split

moved = removed = 0
for d in (IMG / "train", IMG / "val", IMG / "test"):
    if not d.is_dir():
        continue
    for f in list(d.iterdir()):
        target = want.get(f.name)
        if target is None:
            f.unlink(); removed += 1                           # in no split (or a broken .part file)
        elif d.name != target:
            dest = IMG / target / f.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            if dest.exists():
                f.unlink()
            else:
                shutil.move(str(f), str(dest)); moved += 1
print(f"already downloaded: {moved} moved to their new split, {removed} removed")

failed = []
for name, ids in split.items():
    coco = SRC[name]
    jobs = [(coco.imgs[i]["coco_url"], IMG / name / coco.imgs[i]["file_name"]) for i in ids]
    todo = [j for j in jobs if not (j[1].exists() and j[1].stat().st_size > 0)]
    with ThreadPoolExecutor(DOWNLOAD_THREADS) as pool_:
        futures = {pool_.submit(fetch, url, dest): url for url, dest in todo}
        for f in tqdm(as_completed(futures), total=len(futures), desc=f"{name} images", mininterval=10):
            if f.exception() is not None:
                failed.append(futures[f])
    print(f"{name}: {len(jobs) - len(todo)} already here, {len(todo)} downloaded")
if failed:
    print(f"\n{len(failed)} images failed, e.g. {failed[:3]}. Run this cell again; it only fetches what is missing.")
else:
    print("all images present")

/home/jovyan/vault/Kontur/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


already downloaded: 0 moved to their new split, 0 removed


train images: 0it [00:00, ?it/s]


train: 116266 already here, 0 downloaded


val images: 0it [00:00, ?it/s]


val: 1000 already here, 0 downloaded


test images: 0it [00:00, ?it/s]

test: 5000 already here, 0 downloaded
all images present


## Step 4: write YOLO labels and the dataset file

One `.txt` per image: `class cx cy w h`, normalised to 0–1, class = 0..79 in COCO's order.
Old labels and YOLO's label caches are deleted first, so nothing stale is reused.

In [6]:
if LBL.exists():
    shutil.rmtree(LBL)
idx = {cid: k for k, cid in enumerate(cat_ids.values())}     # COCO category id -> 0..79

def write_labels(name, coco, ids):
    out = LBL / name
    out.mkdir(parents=True, exist_ok=True)
    n_obj = 0
    for i in ids:
        im = coco.imgs[i]
        W, H = im["width"], im["height"]
        lines = []
        for a in coco.imgToAnns.get(i, []):
            if a["iscrowd"] or a["category_id"] not in idx:
                continue
            x, y, w, h = a["bbox"]
            x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
            if x2 - x1 < 1 or y2 - y1 < 1:
                continue
            lines.append(f"{idx[a['category_id']]} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} "
                         f"{(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")
        (out / (Path(im["file_name"]).stem + ".txt")).write_text("\n".join(lines) + ("\n" if lines else ""))
        n_obj += len(lines)
    return n_obj

for name, ids in split.items():
    print(f"{name}: {write_labels(name, SRC[name], ids)} objects labelled")

DATA_YAML.write_text(
    f"path: {DATA_DIR.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n"
    + "".join(f"  {k}: {json.dumps(n)}\n" for k, n in enumerate(CLASSES)))
print(DATA_YAML.read_text()[:300], "...")

train: 842790 objects labelled
val: 7112 objects labelled
test: 36334 objects labelled
path: /home/jovyan/vault/Kontur/training/data
train: images/train
val: images/val
test: images/test
names:
  0: "person"
  1: "bicycle"
  2: "car"
  3: "motorcycle"
  4: "airplane"
  5: "bus"
  6: "train"
  7: "truck"
  8: "boat"
  9: "traffic light"
  10: "fire hydrant"
  11: "stop sign"
  12: "par ...


## Training settings (shared by B and C)

Identical for both runs; only the starting weights differ. Each run's best epoch is chosen on
the **val** split.

In [7]:
def train_args(tag, pretrained, **extra):
    a = dict(data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
             workers=WORKERS, device=DEVICE, seed=SEED, deterministic=True, pretrained=pretrained,
             fraction=TRAIN_FRACTION,
             project=str(RUNS), name=tag, exist_ok=True, plots=True)
    a.update(extra)
    return a

JOBS = {   # tag: (start, pretrained)
    "B_scratch": ("yolo11n.yaml", False),       # architecture only: every weight random
    "C_finetune": (str(PRETRAINED), True),      # copy of Kontur's pretrained weights
}
JOBS = {k: v for k, v in JOBS.items() if (k == "B_scratch" and RUN_B) or (k == "C_finetune" and RUN_C)}

def alive(pid):
    """Is the background training process with this id still running?"""
    try:
        os.kill(pid, 0); return True
    except OSError:
        return False

print("will train:", list(JOBS))
print("(after a kernel restart, run Settings, Environment and this cell; then Status or Finish work)")

will train: ['B_scratch', 'C_finetune']
(after a kernel restart, run Settings, Environment and this cell; then Status or Finish work)


## Quick check (optional, about 5 minutes; recommended before the long run)

One epoch of B and C on 1% of the training images, in `runs/check_*` (thrown away afterwards).
If both finish and print metrics, the long run will work. Skip it if you are short on time.

In [ ]:
from ultralytics import YOLO

for tag, (start, pretrained) in JOBS.items():
    t0 = time.perf_counter()
    YOLO(start).train(**train_args(f"check_{tag}", pretrained, epochs=1, patience=1,
                                   fraction=QUICK_CHECK_FRACTION, plots=False))
    print(f"\nquick check {tag}: OK in {time.perf_counter() - t0:.0f} s\n")

New https://pypi.org/project/ultralytics/8.4.220 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.187 🚀 Python-3.11.13 torch-2.14.1+cu130 CUDA:0 (NVIDIA RTX A6000, 48539MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/vault/Kontur/training/data/coco80.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.01, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=tra

Traceback (most recent call last):
  File "/usr/lib/python3.11/multiprocessing/queues.py", line 244, in _feed
    obj = _ForkingPickler.dumps(obj)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^
Traceback (most recent call last):
  File "/usr/lib/python3.11/multiprocessing/reduction.py", line 51, in dumps
    cls(buf, protocol).dump(obj)
  File "/home/jovyan/vault/Kontur/.venv/lib/python3.11/site-packages/torch/multiprocessing/reductions.py", line 613, in reduce_storage
    fd, size = storage._share_fd_cpu_()
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/vault/Kontur/.venv/lib/python3.11/site-packages/torch/storage.py", line 458, in wrapper
    return fn(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
Traceback (most recent call last):
  File "/usr/lib/python3.11/multiprocessing/queues.py", line 244, in _feed
    obj = _ForkingPickler.dumps(obj)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/vault/Kontur/.venv/lib/python3.11/site-packages/torch/storage.py", l

## Launch B and C (in the background)

Each run is a separate background process writing to `runs/<tag>.log`; they keep going if you
close the browser. With `PARALLEL = True`, C starts once B has built YOLO's label caches (so the
two never write the same cache file at once), then both share the GPU.

**Time:** B and C together need roughly 10–20 h for 50 epochs on one A6000, depending on how
fast the data lab's CPUs load images. *Status* prints the real estimate after the first epoch.

In [ ]:
import subprocess

RUNNER = r"""
import json, sys
from ultralytics import YOLO
job = json.loads(sys.argv[1])
if job["resume"]:
    YOLO(job["last"]).train(resume=True)
else:
    YOLO(job["start"]).train(**job["args"])
"""

def launch(tag, start, pretrained):
    last = RUNS / tag / "weights" / "last.pt"
    job = {"start": start, "resume": bool(RESUME and last.exists()), "last": str(last),
           "args": train_args(tag, pretrained)}
    log = open(RUNS / f"{tag}.log", "a")
    log.write(f"\n===== launch {time.strftime('%Y-%m-%d %H:%M:%S')} resume={job['resume']} =====\n"); log.flush()
    p = subprocess.Popen([sys.executable, "-c", RUNNER, json.dumps(job)], cwd=str(HERE),
                         stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    (RUNS / f"{tag}.pid").write_text(str(p.pid))
    print(f"{tag}: started (pid {p.pid}), log: runs/{tag}.log")
    return p

caches = [LBL / "train.cache", LBL / "val.cache"]
if not RESUME:
    # The quick check leaves caches for 1% of the data. Delete them, so the first run builds
    # the full ones and the second waits for them (two runs never write one cache at once).
    for c in caches:
        c.unlink(missing_ok=True)
procs = {}
for k, (tag, (start, pretrained)) in enumerate(JOBS.items()):
    if PARALLEL and k > 0:
        t0 = time.time()                          # wait for the first run to build the label caches
        while not all(c.exists() for c in caches) and time.time() - t0 < 1800:
            if not alive(procs[next(iter(procs))].pid):
                raise RuntimeError("the first run stopped early; read its log in runs/")
            time.sleep(10)
        time.sleep(30)
    if not PARALLEL and k > 0:
        print(f"waiting for {list(procs)[-1]} to finish (PARALLEL = False) ...")
        procs[list(procs)[-1]].wait()
    procs[tag] = launch(tag, start, pretrained)

## Status (re-run any time)

In [ ]:
import pandas as pd

def status():
    rows = []
    for tag in JOBS:
        pid_f, res_f, log_f = RUNS / f"{tag}.pid", RUNS / tag / "results.csv", RUNS / f"{tag}.log"
        pid = int(pid_f.read_text()) if pid_f.exists() else None
        done, secs, best = 0, None, None
        if res_f.exists():
            d = pd.read_csv(res_f); d.columns = [c.strip() for c in d.columns]
            done = len(d)
            if done:
                secs = float(d["time"].iloc[-1]) if "time" in d else None
                best = float(d["metrics/mAP50-95(B)"].max())
        left = (secs / done * (EPOCHS - done) / 3600) if (secs and done) else None
        rows.append({"run": tag, "running": bool(pid and alive(pid)), "epochs": f"{done}/{EPOCHS}",
                     "hours so far": round(secs / 3600, 2) if secs else None,
                     "hours left (est.)": round(left, 1) if left is not None else None,
                     "best val mAP50-95": round(best, 4) if best is not None else None})
        if pid and not alive(pid) and done < EPOCHS and log_f.exists():
            print(f"--- {tag} is not running and not finished; end of its log:")
            print("".join(log_f.read_text(errors="replace").splitlines(True)[-15:]))
    return pd.DataFrame(rows).set_index("run")

status()

## Finish: keep the evidence

Run when *Status* shows every run at its last epoch and not running. Copies each run's best
weights (chosen on **val**) to `trained/`, and its curves and settings to `results/`.

In [ ]:
import pandas as pd

for tag in JOBS:
    best = RUNS / tag / "weights" / "best.pt"
    if not best.exists():
        print(f"{tag}: no best.pt yet"); continue
    dest = TRAINED / f"yolo11n_{tag}_coco.pt"
    shutil.copy(best, dest)
    for f in ("results.csv", "results.png", "confusion_matrix.png", "confusion_matrix_normalized.png", "args.yaml"):
        if (RUNS / tag / f).exists():
            shutil.copy(RUNS / tag / f, RESULTS / f"{tag}_{f}")
    d = pd.read_csv(RUNS / tag / "results.csv"); d.columns = [c.strip() for c in d.columns]
    env.setdefault("training", {})[tag] = {
        "epochs": len(d), "hours": round(float(d["time"].iloc[-1]) / 3600, 2) if "time" in d else None,
        "best_val_mAP50_95": round(float(d["metrics/mAP50-95(B)"].max()), 4)}
    print(f"{tag}: {dest.name}  {env['training'][tag]}")
env["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
(RESULTS / "environment.json").write_text(json.dumps(env, indent=1))
print("\nNext: eval/04_scratch_vs_pretrained.ipynb")